In [1]:
import multiprocessing
import os
import re
from concurrent.futures import ProcessPoolExecutor, as_completed
import gc
import pandas as pd
import matplotlib.pyplot as plt
from pyulog import ULog

In [2]:
# return the array of combined log file line
def getLogData(baseLogDir, iteration, testCase, model="iris"):
    logPath = os.path.join(baseLogDir, iteration, model, testCase, "log-combined.log_plain.log")
    if os.path.exists(logPath):
        with open(logPath, "r") as f:
            return f.readlines()
    return None

In [3]:
# return the ulg file name parsed from the combined log
def findUlgName(log):
    pattern = r"INFO\s+\[logger\]\s+Opened full log file:\s+(.*\.ulg)"
    match = re.search(pattern, log)
    return match.group(1) if match else ""

In [4]:
# return the ulog parsed from the ulg file
def getUlogData(baseUlgDir, ulgFileName):
    normPath = os.path.normpath(ulgFileName)
    ulgPath = os.path.join(baseUlgDir, normPath)
    return ULog(ulgPath) if os.path.exists(ulgPath) else None

In [5]:
def process_test_case(args):
    baseLogDir, baseUlgDir, testIteration, model, testCase = args

    try:
        # 1) combined log 읽기
        combinedLog = getLogData(baseLogDir, testIteration, testCase, model)
        ulgFileName = findUlgName("".join(combinedLog) if combinedLog else "")
        # 2) ULog 읽기
        ulog = getUlogData(baseUlgDir, ulgFileName)
        # 3) estimator_innovation_variance에서 gps_hpos 추출
        innov_df = pd.DataFrame(ulog.get_dataset("estimator_innovation_variances").data)
        # 4) actuator_armed에서 Arm 시점(timestamp) 추출
        armed_df = pd.DataFrame(ulog.get_dataset("actuator_armed").data)
        arm_events = armed_df[armed_df["armed"] == 1]
        arm_time = arm_events["timestamp"].iloc[0] if not arm_events.empty else None

        # 5) Arm 이후부터 로그 마지막까지 gps_hpos 통계 계산
        if arm_time is not None:
            flight_df = innov_df[innov_df["timestamp"] >= arm_time]
        else:
            flight_df = innov_df

        gps_Xseries = flight_df["gps_hpos[0]"]
        gps_Yseries = flight_df["gps_hpos[0]"]
        mean_Xgps = gps_Xseries.mean()
        mean_Ygps = gps_Xseries.mean()
        max_Xgps  = gps_Xseries.max()
        max_Ygps  = gps_Yseries.max()
        min_Xgps  = gps_Xseries.min()
        min_Ygps  = gps_Yseries.min()

        # 메모리 정리
        del ulog, innov_df, armed_df, flight_df
        gc.collect()

        return testIteration, testCase, {
            "gps_hpos[0]": {
                "mean": mean_Xgps,
                "max": max_Xgps,
                "min": min_Xgps
            },
            "gps_hpos[1]": {
                "mean": mean_Ygps,
                "max": max_Ygps,
                "min": min_Ygps
            }
        }

    except Exception as e:
        print(f"Error processing {testIteration}/{model}/{testCase}: {e}")
        return testIteration, testCase, {"error": str(e)}

In [6]:
def load_flight_data_parallel(baseLogDir, baseUlgDir, max_workers=None):
    if max_workers is None:
        max_workers = max(1, multiprocessing.cpu_count() // 2)

    flightData = {}
    tasks = []

    for testIteration in os.listdir(baseLogDir):
        iterationDir = os.path.join(baseLogDir, testIteration)
        flightData[testIteration] = {}

        for model_name in os.listdir(iterationDir):
            modelDir = os.path.join(iterationDir, model_name)

            for testCase in os.listdir(modelDir):
                # testCase가 normal*인 경우만 처리 (대소문자 구분 없이)
                if not testCase.lower().startswith("normal"):
                    continue
                tasks.append((baseLogDir, baseUlgDir, testIteration, model_name, testCase))

    total, done = len(tasks), 0
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_task = {executor.submit(process_test_case, t): t for t in tasks}
        for future in as_completed(future_to_task):
            it, tc, result = future.result()
            flightData[it][tc] = result
            done += 1
            if done % 10 == 0 or done == total:
                print(f"Progress: {done}/{total} ({done/total*100:.1f}%)")

    return flightData

In [8]:
# ulg, combined log 기본 위치
baseLogDir = os.path.expanduser("~/ws/PX4-Autopilot/logs/2025-03-21T21-42-37Z")
baseUlgDir = os.path.expanduser("~/ws/PX4-Autopilot/build/px4_sitl_default/tmp_mavsdk_tests/rootfs")

# 병렬 처리를 통한 데이터 로딩
flightData = load_flight_data_parallel(baseLogDir, baseUlgDir)

# 결과 확인
print("Data loading completed.")

# 모든 비행의 gps_hpos[0], gps_hpos[1] 통계 수집
stats = []
for it, cases in flightData.items():
    for tc, res in cases.items():
        if "error" in res:
            continue
        # X축 (gps_hpos[0])
        stats.append({
            "iteration": it,
            "testCase": tc,
            "field": "gps_hpos[0]",
            **res["gps_hpos[0]"]
        })
        # Y축 (gps_hpos[1])
        stats.append({
            "iteration": it,
            "testCase": tc,
            "field": "gps_hpos[1]",
            **res["gps_hpos[1]"]
        })

stats_df = pd.DataFrame(stats)

# 축별 전체 집계 통계 출력
for field in ["gps_hpos[0]", "gps_hpos[1]"]:
    df_f = stats_df[stats_df["field"] == field]
    overall_mean = df_f["mean"].mean()
    overall_max  = df_f["max"].max()
    overall_min  = df_f["min"].min()

    print(f"=== 전체 비행 {field} 통계 ===")
    print(f"Average of means: {overall_mean:.3f}")
    print(f"Maximum of maxima: {overall_max:.3f}")
    print(f"Minimum of minima: {overall_min:.3f}")


Progress: 10/400 (2.5%)
Progress: 20/400 (5.0%)
Progress: 30/400 (7.5%)
Progress: 40/400 (10.0%)
Progress: 50/400 (12.5%)
Progress: 60/400 (15.0%)
Progress: 70/400 (17.5%)
Progress: 80/400 (20.0%)
Progress: 90/400 (22.5%)
Progress: 100/400 (25.0%)
Progress: 110/400 (27.5%)
Progress: 120/400 (30.0%)
Progress: 130/400 (32.5%)
Progress: 140/400 (35.0%)
Progress: 150/400 (37.5%)
Progress: 160/400 (40.0%)
Progress: 170/400 (42.5%)
Progress: 180/400 (45.0%)
Progress: 190/400 (47.5%)
Progress: 200/400 (50.0%)
Progress: 210/400 (52.5%)
Progress: 220/400 (55.0%)
Progress: 230/400 (57.5%)
Progress: 240/400 (60.0%)
Progress: 250/400 (62.5%)
Progress: 260/400 (65.0%)
Progress: 270/400 (67.5%)
Progress: 280/400 (70.0%)
Progress: 290/400 (72.5%)
Progress: 300/400 (75.0%)
Progress: 310/400 (77.5%)
Progress: 320/400 (80.0%)
Progress: 330/400 (82.5%)
Progress: 340/400 (85.0%)
Progress: 350/400 (87.5%)
Progress: 360/400 (90.0%)
Progress: 370/400 (92.5%)
Progress: 380/400 (95.0%)
Progress: 390/400 (97.5%